In [0]:
-- Crear la tabla vacía si no existe
CREATE TABLE IF NOT EXISTS `brazillian-e-commerce`.`02_silver`.orders
USING DELTA
AS
SELECT
    order_id,
    customer_id,
    order_status,
    CAST(order_purchase_timestamp AS TIMESTAMP) AS purchase_ts,
    CAST(order_approved_at AS TIMESTAMP) AS approved_ts,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered_ts,
    CAST(order_estimated_delivery_date AS DATE) AS estimated_dt
FROM `brazillian-e-commerce`.`03_bronze`.orders
WHERE 1 = 0;

-- Carga inicial e incremental
MERGE INTO `brazillian-e-commerce`.`02_silver`.orders AS target
USING (
    SELECT
        order_id,
        customer_id,
        order_status,
        CAST(order_purchase_timestamp AS TIMESTAMP) AS purchase_ts,
        CAST(order_approved_at AS TIMESTAMP) AS approved_ts,
        CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered_ts,
        CAST(order_estimated_delivery_date AS DATE) AS estimated_dt
    FROM `brazillian-e-commerce`.`03_bronze`.orders
) AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
    UPDATE SET *

WHEN NOT MATCHED THEN
    INSERT *;

In [0]:
CREATE TABLE IF NOT EXISTS `brazillian-e-commerce`.`02_silver`.orderItems
USING DELTA
AS 
SELECT
    order_id,
    order_item_id,
    product_id,
    seller_id,
    CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
    CAST(price AS DECIMAL(10,2)) AS price,
    CAST(freight_value AS DECIMAL(10,2)) AS freight_value

FROM `brazillian-e-commerce`.`03_bronze`.orderItems
WHERE 1=0;

MERGE INTO `brazillian-e-commerce`.`02_silver`.orderItems AS target
USING (
    SELECT
        order_id,
        order_item_id,
        product_id,
        seller_id,
        CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
        CAST(price AS DECIMAL(10,2)) AS price,
        CAST(freight_value AS DECIMAL(10,2)) AS freight_value
    FROM `brazillian-e-commerce`.`03_bronze`.orderItems
) AS source

ON target.order_id = source.order_id 
AND target.order_item_id = source.order_item_id
WHEN MATCHED THEN 
    UPDATE SET *
WHEN NOT MATCHED THEN 
    INSERT *;
        

In [0]:
CREATE TABLE IF NOT EXISTS `brazillian-e-commerce`.`02_silver`.products
USING DELTA
AS 
SELECT
    product_id,
    product_category_name,
    CAST(product_name_lenght AS INT) AS product_name_length,
    CAST(product_description_lenght AS INT) AS product_description_length,
    CAST(product_photos_qty AS INT) AS product_photos_qty,
    CAST(product_weight_g AS INT) AS product_weight_g,
    CAST(product_length_cm AS INT) AS product_length_cm,
    CAST(product_height_cm AS INT) AS product_height_cm,
    CAST(product_width_cm AS INT) AS product_width_cm

FROM `brazillian-e-commerce`.`03_bronze`.products
WHERE 1=0;

MERGE INTO `brazillian-e-commerce`.`02_silver`.products AS target
USING (
    SELECT
        product_id,
        product_category_name,
        CAST(product_name_lenght AS INT) AS product_name_length,
        CAST(product_description_lenght AS INT) AS product_description_length,
        CAST(product_photos_qty AS INT) AS product_photos_qty,
        CAST(product_weight_g AS INT) AS product_weight_g,
        CAST(product_length_cm AS INT) AS product_length_cm,
        CAST(product_height_cm AS INT) AS product_height_cm,
        CAST(product_width_cm AS INT) AS product_width_cm
    FROM `brazillian-e-commerce`.`03_bronze`.products
) AS source

ON target.product_id = source.product_id 

WHEN MATCHED THEN 
    UPDATE SET *
WHEN NOT MATCHED THEN 
    INSERT * ;

In [0]:
CREATE TABLE IF NOT EXISTS `brazillian-e-commerce`.`02_silver`.sellers
USING DELTA
AS 
SELECT
    CAST(seller_id AS STRING) AS seller_id, 
    CAST(seller_zip_code_prefix AS STRING) AS seller_zip_code_prefix,
    CAST(seller_city AS STRING) AS seller_city,
    CAST(seller_state AS STRING) AS seller_state 

FROM `brazillian-e-commerce`.`03_bronze`.sellers
WHERE 1=0;

MERGE INTO `brazillian-e-commerce`.`02_silver`.sellers AS target
USING (
    SELECT
        CAST(seller_id AS STRING) AS seller_id, 
        CAST(seller_zip_code_prefix AS STRING) AS seller_zip_code_prefix,
        CAST(seller_city AS STRING) AS seller_city,
        CAST(seller_state AS STRING) AS seller_state
  
    FROM `brazillian-e-commerce`.`03_bronze`.sellers
) AS source

ON target.seller_id = source.seller_id

WHEN MATCHED THEN 
    UPDATE SET *
WHEN NOT MATCHED THEN 
    INSERT * ;

In [0]:
CREATE TABLE IF NOT EXISTS `brazillian-e-commerce`.`02_silver`.customers
USING DELTA
AS 
SELECT
    CAST(customer_id AS STRING) AS customer_id,
    CAST(customer_unique_id AS STRING) AS customer_unique_id,
    CAST(customer_zip_code_prefix AS STRING) AS customer_zip_code_prefix,
    CAST(customer_city AS STRING) AS customer_city,
    CAST(customer_state AS STRING) AS customer_state
    
 FROM `brazillian-e-commerce`.`03_bronze`.customers
WHERE 1=0;

MERGE INTO `brazillian-e-commerce`.`02_silver`.customers AS target
USING (
    SELECT
      CAST(customer_id AS STRING) AS customer_id,
      CAST(customer_unique_id AS STRING) AS customer_unique_id,
      CAST(customer_zip_code_prefix AS STRING) AS customer_zip_code_prefix,
      CAST(customer_city AS STRING) AS customer_city,
      CAST(customer_state AS STRING) AS customer_state
  
    FROM `brazillian-e-commerce`.`03_bronze`.customers
) AS source

ON target.customer_unique_id = source.customer_unique_id
AND target.customer_id = source.customer_id

WHEN MATCHED THEN 
    UPDATE SET *
WHEN NOT MATCHED THEN 
    INSERT * ;

In [0]:
CREATE OR REPLACE TABLE `brazillian-e-commerce`.02_silver.geolocation
AS

SELECT DISTINCT
    geolocation_zip_code_prefix,

    CAST(geolocation_lat AS DECIMAL(10,8)) AS geolocation_lat,
    CAST(geolocation_lng AS DECIMAL(11,8)) AS geolocation_lng,
    TRIM(UPPER(geolocation_city)) AS geolocation_city,
    UPPER(geolocation_state) AS geolocation_state

FROM `brazillian-e-commerce`.03_bronze.geolocation;

In [0]:
CREATE OR REPLACE TABLE `brazillian-e-commerce`.02_silver.orderPayments
AS

SELECT
    order_id,

    CAST(payment_sequential AS INT) AS payment_sequential,
    UPPER(TRIM(payment_type)) AS payment_type,
    CAST(payment_installments AS INT) AS payment_installments,
    CAST(payment_value AS DECIMAL(10,2)) AS payment_value

FROM `brazillian-e-commerce`.03_bronze.orderPayments;

In [0]:
CREATE OR REPLACE TABLE `brazillian-e-commerce`.02_silver.orderReviews
AS

SELECT
    review_id,
    order_id,

    CAST(review_score AS INT) AS review_score,
    TRIM(review_comment_title) AS review_comment_title,
    TRIM(review_comment_message) AS review_comment_message,
    CAST(review_creation_date AS DATE) AS review_creation_date,
    CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_ts

FROM `brazillian-e-commerce`.03_bronze.orderReviews;